# Test Analysis 2025

In [ ]:
import json
import re
import numpy as np
import pandas as pd
import seaborn as sns
import glob
from IPython.display import Markdown, display
import pygwalker as pyg
from pygwalker.api.streamlit import StreamlitRenderer
import streamlit as st
import webbrowser
import matplotlib.pyplot as plt
import ipywidgets as widgets
import plotly.express as px
import plotly.graph_objects as go
from ipywidgets import interact, Dropdown, ToggleButtons, SelectMultiple, interactive, fixed, interact_manual, VBox, HBox, Label
%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')

## Data Preparation

In [ ]:
from data_processor import load_test_data, load_defect_data
tdf = load_test_data()
ddf = load_defect_data()
print(f"测试数据: {tdf.shape}, 缺陷数据: {ddf.shape}")

In [ ]:
pyg.walk(tdf)

In [ ]:
tdf['project']=''
tdf.loc[tdf.name.str.contains('DTSV_CHINA-RSU') & (tdf.project==''), 'project'] = 'RSU'
tdf.loc[tdf.name.str.contains('RSU') & (tdf.project==''), 'project'] = 'RSU'
tdf.loc[tdf.name.str.contains('IOS') & (tdf.project==''), 'project'] = 'App'
tdf.loc[tdf.name.str.contains('Android') & (tdf.project==''), 'project'] = 'App'
tdf.loc[tdf.name.str.contains('HarmonyOS') & (tdf.project==''), 'project'] = 'App'
tdf.loc[tdf.name.str.contains('IDC23_MINI') & (tdf.project==''), 'project'] = 'IDC'
tdf.loc[tdf.name.str.contains('IDC23_BMW') & (tdf.project==''), 'project'] = 'IDC'
tdf.loc[tdf.name.str.contains('HU-MGU_02_A') & (tdf.project==''), 'project'] = 'IDC'
tdf.loc[tdf.name.str.contains('IDC23') & (tdf.project==''), 'project'] = 'IDC'
tdf.loc[tdf.name.str.contains('MGU22') & (tdf.project==''), 'project'] = 'MGU22'
tdf.loc[tdf.name.str.contains('HU-MGU_02_L') & (tdf.project==''), 'project'] = 'MGU22'
tdf.loc[tdf.name.str.contains('HU-MGU_01') & (tdf.project==''), 'project'] = 'MGU22' # What is MGU_01?
tdf.loc[tdf.name.str.contains('MGU21') & (tdf.project==''), 'project'] = 'MGU21'
tdf.loc[tdf.name.str.contains('MGU18') & (tdf.project==''), 'project'] = 'MGU18'
tdf.loc[tdf.name.str.contains('IDCEVO') & (tdf.project==''), 'project'] = 'IDCevo'
tdf.loc[tdf['project'].str.startswith('MGU', na=False), 'project'] = 'MGU'


In [ ]:
tdf["model"] = tdf["exec_model_series_udf"].apply(lambda x: x["name"] if x!=None else "")
tdf["test_id"] = tdf["test"].apply(lambda x: x["id"] if x!=None else "")
tdf["test_name"] = tdf["test"].apply(lambda x: x["name"] if x!=None else "")
tdf["author_name"] = tdf["author"].apply(lambda x: x["full_name"] if x!=None else "")
tdf["test_event"] = tdf["release"].apply(lambda x: x["name"] if x!=None else "")
tdf["tester"] = tdf["run_by"].apply(lambda x: x["full_name"] if x!=None else "")
tdf["aida_count"] = tdf["product_areas"].apply(lambda x: x["total_count"] if x!=None else "")
tdf["aida"] = tdf["product_areas"].apply(lambda x: [i["name"] for i in x["data"]] if x!=None else "")
tdf["pu"] = tdf["set_udf"].apply(lambda x: x["name"] if x!=None else "")
tdf["run_status"] = tdf["status"].apply(lambda x: x["name"] if x!=None else "")

In [ ]:
tdf['finished_udf'] = tdf['finished_udf'].str.replace('T', ' ').str.replace('Z', '')
tdf['finished_udf'] = pd.to_datetime(tdf['finished_udf'], format='%Y-%m-%d %H:%M:%S')
tdf['test_week'] = tdf['test_event'] + 'CW' + tdf['finished_udf'].dt.isocalendar().week.astype("string").str.zfill(2)

In [ ]:
print("所有项目列表：", tdf['project'].unique())

In [ ]:
def get_top_req(req):
    lst_req = req
    top_req = ''
    len_req = 1000
    n = len(lst_req)
    for i in range(n):
        ith_req = lst_req[i]
        len_ith_req = len(ith_req.split(' ')[-1])
        if (len_ith_req < len_req):
            len_req = len_ith_req
            top_req = ith_req
    return top_req
tdf["top_aida"] = tdf["aida"].apply(get_top_req)

##### Create *FV*

In [ ]:
excel_file = "aida/top_aida_project_fv_mapping.xlsx"
writer = pd.ExcelFile(excel_file)

for sheet_name in writer.sheet_names:
    df = pd.read_excel(excel_file, sheet_name=sheet_name)
    fv_col = f"fv_{sheet_name}"
    df = df.rename(columns={'fv': fv_col})
    df_subset = df[['project', 'top_aida', fv_col]]
    tdf = tdf.merge(df_subset, on=['project', 'top_aida'], how='left')

fv_columns = [col for col in tdf.columns if col.startswith('fv_')]
tdf['fv'] = tdf[fv_columns].bfill(axis=1).iloc[:, 0]
tdf.drop(columns=fv_columns, inplace=True)

##### Create *Team*

In [ ]:
dips_fvs = [
    'DIPS_TSP_Call_Services', 'DIPS_TSP_CD_Updates', 'DIPS_TSP_Remote_Services',
    'Mybmw App', 'eMob', 'DIPS_TSP_Car_Apps_CN', 'DIPS_TSP_MobileApps',
    'DIPS_TSP_Enabler', 'Slip-Through' # 修改后的值
]

tdf['team'] = np.where(
    (tdf['fv'].isin(dips_fvs)) | (tdf['fv'].isna()), # 修改条件，增加对 isna() 的检查
    'DIPS',
    'IUK'
)

##### Create *FVP*

In [ ]:
fvp_mapping = {
    'DIPS_TSP_Call_Services': 'Tianhua',
    'DIPS_TSP_CD_Updates': 'Tianhua',
    'DIPS_TSP_Remote_Services': 'Tianhua',
    'eMob': 'Tianhua',
    'DIPS_TSP_MobileApps': 'Tianhua',
    'DIPS_TSP_Enabler': 'Tianhua', 
    'IuK_TSP_Navi': 'Tony',
    'IuK_TSP_AZV': 'Xu Miao',
    'IuK_TSP_Entertainment': 'Xu Miao',
    'IuK_TSP_Audio': 'Xu Miao',
    'IuK_TSP_Connectivity': 'Xu Miao',
    'DIPS_TSP_Car_Apps_CN': 'Huanran',
    'IuK_TSP_HMI': 'Jerry',
    'DIPS_TSP_RSU': 'Jerry',
    'IuK_TSP_Carfunctions': 'Jerry',
    'IuK_TSP_Perso CN': 'Jerry',
    'RSU': 'Jerry',
    'Mybmw App': 'Marin'
    # 注意：'DIPS_TSP_Enabler Slip-Through' 如果也需要映射，也要添加
}

tdf['fvp'] = tdf['fv'].map(fvp_mapping)
tdf['fvp'] = tdf['fvp'].fillna('Unknown')

In [ ]:
# 假设 tdf 是通过 load_test_data() 加载的 DataFrame
if 'test_name' in tdf.columns:
    # 确保 test_name 列是字符串类型，处理 NaN (如果尚未处理)
    tdf['test_name'] = tdf['test_name'].fillna('').astype(str)

    print("\n正在检查 'test_name' 字段中包含 'RSU' (忽略大小写) 的值...")
    rsu_test_names_mask = tdf['test_name'].str.contains('RSU', na=False, case=False)
    rsu_test_names = tdf.loc[rsu_test_names_mask, 'test_name'].unique()

    if len(rsu_test_names) > 0:
        print(f"找到 {len(rsu_test_names)} 个包含 'RSU' 的唯一 'test_name' 字段:")
        for name_val in rsu_test_names:
            print(f"- {name_val}")
    else:
        print("在加载的测试数据的 'test_name' 字段中未找到任何包含 'RSU' 的值。")
    print("--- 检查完毕 ---\n")
else:
    print("警告: DataFrame 中缺少 'test_name' 列，无法检查 RSU 相关测试名称。")


In [ ]:
plt.figure(figsize=(10, 6))
sns.heatmap(tdf.isnull(), cbar=False, cmap='viridis')
plt.title("Missing Values Heatmap")
plt.show()

missing_stats = tdf.isnull().sum().sort_values(ascending=False)
missing_stats = missing_stats[missing_stats > 0]
print("缺失值统计:\n", missing_stats)


## Insights

### Pygwalker

In [ ]:
pyg.walk(tdf)

In [ ]:
html = pyg.to_html(tdf)
with open("pygwalker_output.html", "w") as f:
    f.write(html)
webbrowser.open("pygwalker_output.html")

### Profiling

In [ ]:
from ipywidgets import interact, Dropdown, HBox, VBox, Layout, Output, SelectMultiple, ToggleButtons, HTML
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display
import pandas as pd
import numpy as np

# 准备筛选器选项
projects = ['All'] + sorted(tdf['project'].dropna().unique().tolist())
fvs = ['All'] + sorted(tdf['fv'].dropna().unique().tolist())
teams = ['All'] + sorted(tdf['team'].dropna().unique().tolist())
testers = ['All'] + sorted(tdf['tester'].dropna().unique().tolist())
aidas = ['All'] + sorted(tdf['top_aida'].dropna().unique().tolist())
test_weeks = ['All'] + sorted(tdf['test_week'].dropna().unique().tolist())
releases = ['All'] + sorted(tdf['test_event'].dropna().unique().tolist())
run_statuses = ['All', 'Passed', 'Failed', 'Blocked', 'Planned']

# 自定义颜色方案
color_palette = {
    'background': '#f5f5f5',
    'panel': '#ffffff',
    'text': '#333333',
    'primary': '#1f77b4',
    'secondary': '#ff7f0e',
    'success': '#2ca02c',
    'danger': '#d62728',
    'warning': '#ffbb78'
}

# 创建标题
def create_header():
    display(HTML("""
    <div style="background-color:#2c3e50; color:white; padding:15px; margin-bottom:20px; border-radius:5px;">
        <h1 style="margin:0;">DTSV Test Quality</h1>
    </div>
    """))

# 创建侧边栏 - 现在返回一个HTML widget而不是直接显示
def create_sidebar():
    sidebar = HTML("""
    <div style="background-color:#ecf0f1; padding:15px; border-radius:5px; margin-right:20px; width:250px;">
        <h3 style="border-bottom:1px solid #bdc3c7; padding-bottom:10px;">Defect Status</h3>
        <ul style="list-style-type:none; padding-left:10px;">
            <li><b>Team Velocity</b></li>
            <li><b>Test Quality</b>
                <ul style="list-style-type:none; padding-left:15px;">
                    <li>Cash Quality</li>
                    <li>Supplier Quality</li>
                    <li>Octane</li>
                    <li>Defect Panel</li>
                    <li>OBS</li>
                    <li>LookUp Table</li>
                </ul>
            </li>
        </ul>
        
        <h3 style="border-bottom:1px solid #bdc3c7; padding-bottom:10px; margin-top:20px;">Global Dips</h3>
        <ul style="list-style-type:none; padding-left:10px;">
            <li>Gideal Dips</li>
            <li>HMI</li>
            <li>Entertainment</li>
            <li>Com/Audio</li>
            <li>Local Dips</li>
            <li>Pad</li>
            <li>Navi</li>
            <li>ACV/Kombi</li>
            <li>RSU</li>
        </ul>
    </div>
    """)
    return sidebar  # 返回widget而不是直接display

# 创建更美观的交互控件
def create_dashboard():
    # 创建标题
    create_header()
    
    # 创建侧边栏和主内容区域
    sidebar = create_sidebar()  # 现在返回HTML widget
    main_content = Output()
    
    # 使用HBox布局
    main_layout = HBox([sidebar, main_content], layout=Layout(width='100%'))
    display(main_layout)
    
    with main_content:
        # 创建控制面板
        control_panel = HTML("""
        </div>
        """)
        display(control_panel)
        
        # 创建下拉菜单
        project_dropdown = Dropdown(options=projects, value='All', description='项目:', layout=Layout(width='250px'))
        fv_dropdown = Dropdown(options=fvs, value='All', description='功能:', layout=Layout(width='250px'))
        team_dropdown = Dropdown(options=teams, value='All', description='团队:', layout=Layout(width='250px'))
        tester_dropdown = Dropdown(options=testers, value='All', description='测试员:', layout=Layout(width='250px'))
        aida_dropdown = Dropdown(options=aidas, value='All', description='TOP AIDA:', layout=Layout(width='250px'))
        week_dropdown = Dropdown(options=test_weeks, value='All', description='测试周:', layout=Layout(width='250px'))
        release_dropdown = Dropdown(options=releases, value='All', description='版本:', layout=Layout(width='250px'))
        status_dropdown = Dropdown(options=run_statuses, value='All', description='运行状态:', layout=Layout(width='250px'))

        # 创建输出区域
        trend_output = Output(layout=Layout(width='100%', margin='10px 0'))
        execution_output = Output(layout=Layout(width='100%', margin='10px 0'))
        aida_output = Output(layout=Layout(width='100%', margin='10px 0'))

        # 定义更新函数
        def update_dashboard(change):
            with trend_output:
                trend_output.clear_output()
                with execution_output:
                    execution_output.clear_output()
                with aida_output:
                    aida_output.clear_output()
                
                project = project_dropdown.value
                fv = fv_dropdown.value
                team = team_dropdown.value
                tester = tester_dropdown.value
                aida = aida_dropdown.value
                test_week = week_dropdown.value
                release = release_dropdown.value
                status = status_dropdown.value
                
                # 数据筛选
                filtered = tdf.copy()
                if project != 'All':
                    filtered = filtered[filtered['project'] == project]
                if fv != 'All':
                    filtered = filtered[filtered['fv'] == fv]
                if team != 'All':
                    filtered = filtered[filtered['team'] == team]
                if tester != 'All':
                    filtered = filtered[filtered['tester'] == tester]
                if aida != 'All':
                    filtered = filtered[filtered['top_aida'] == aida]
                if test_week != 'All':
                    filtered = filtered[filtered['test_week'] == test_week]
                if release != 'All':
                    filtered = filtered[filtered['test_event'] == release]
                if status != 'All':
                    filtered = filtered[filtered['run_status'] == status]
                
                # 1. 通过率趋势图
                if not filtered.empty:
                    # 按周计算通过率
                    weekly_data = filtered.groupby('test_week').agg(
                        total_tests=('run_status', 'count'),
                        passed_tests=('run_status', lambda x: (x == 'Passed').sum()))
                    weekly_data['pass_rate'] = (weekly_data['passed_tests'] / weekly_data['total_tests'] * 100).round(2)
                    weekly_data = weekly_data.reset_index()
                    
                    # 创建趋势图
                    fig_trend = go.Figure()
                    fig_trend.add_trace(go.Scatter(
                        x=weekly_data['test_week'], 
                        y=weekly_data['pass_rate'],
                        mode='lines+markers',
                        name='通过率',
                        line=dict(color=color_palette['primary'], width=3),
                        marker=dict(size=10, color=color_palette['primary']),
                        text=weekly_data['pass_rate'].apply(lambda x: f"{x}%"),
                        hoverinfo='text+x+y'
                    ))
                    
                    fig_trend.update_layout(
                        title=f"<b>测试通过率趋势</b>",
                        title_font_size=16,
                        xaxis_title='测试周',
                        yaxis_title='通过率 (%)',
                        height=400,
                        hovermode="x unified",
                        plot_bgcolor=color_palette['panel'],
                        paper_bgcolor=color_palette['background'],
                        font_color=color_palette['text'],
                        margin=dict(l=50, r=50, b=50, t=60, pad=10)
                    )
                else:
                    fig_trend = go.Figure()
                    fig_trend.update_layout(
                        title="<b>无数据</b>",
                        height=400,
                        plot_bgcolor=color_palette['panel'],
                        paper_bgcolor=color_palette['background'],
                        font_color=color_palette['text']
                    )
                
                # 2. 测试用例执行量统计
                if not filtered.empty:
                    # 按状态统计
                    status_counts = filtered['run_status'].value_counts().reset_index()
                    status_counts.columns = ['run_status', 'count']
                    
                    # 创建执行量图表
                    fig_execution = px.bar(
                        status_counts,
                        x='run_status',
                        y='count',
                        color='run_status',
                        title="<b>测试用例执行状态分布</b>",
                        labels={'count': '数量', 'run_status': '运行状态'},
                        height=400,
                        color_discrete_map={
                            'Passed': color_palette['success'],
                            'Failed': color_palette['danger'],
                            'Blocked': color_palette['warning'],
                            'Planned': color_palette['secondary']
                        }
                    )
                    
                    # 添加总数标签和样式调整
                    fig_execution.update_traces(
                        texttemplate='%{y}',
                        textposition='outside',
                        marker_line_width=0
                    )
                    fig_execution.update_layout(
                        title_font_size=16,
                        plot_bgcolor=color_palette['panel'],
                        paper_bgcolor=color_palette['background'],
                        font_color=color_palette['text'],
                        margin=dict(l=50, r=50, b=50, t=60, pad=10),
                        showlegend=False
                    )
                else:
                    fig_execution = go.Figure()
                    fig_execution.update_layout(
                        title="<b>无数据</b>",
                        height=400,
                        plot_bgcolor=color_palette['panel'],
                        paper_bgcolor=color_palette['background'],
                        font_color=color_palette['text']
                    )
                
                # 3. TOP5 AIDA问题分布
                if not filtered.empty and 'top_aida' in filtered.columns:
                    aida_counts = filtered['top_aida'].value_counts().nlargest(5).reset_index()
                    aida_counts.columns = ['top_aida', 'count']
                    
                    # 添加测试用例ID信息
                    aida_details = filtered.groupby('top_aida')['id'].apply(
                        lambda x: "<br>".join([f"ID: {id}" for id in x.unique()[:5]]) + ("<br>..." if len(x.unique()) > 5 else "")
                    ).reset_index()
                    aida_counts = aida_counts.merge(aida_details, on='top_aida', how='left')
                    
                    fig_aida = px.pie(
                        aida_counts,
                        values='count',
                        names='top_aida',
                        title="<b>TOP 5 AIDA问题分布</b>",
                        height=400,
                        hover_data=['id'],
                        color_discrete_sequence=[color_palette['primary'], color_palette['secondary'], 
                                               color_palette['success'], color_palette['warning'], 
                                               color_palette['danger']]
                    )
                    
                    fig_aida.update_traces(
                        textposition='inside',
                        textinfo='percent+label',
                        hovertemplate="<b>%{label}</b><br>数量: %{value}<br>%{customdata}",
                        marker_line_width=0.5,
                        marker_line_color='white'
                    )
                    fig_aida.update_layout(
                        title_font_size=16,
                        plot_bgcolor=color_palette['panel'],
                        paper_bgcolor=color_palette['background'],
                        font_color=color_palette['text'],
                        margin=dict(l=50, r=50, b=50, t=60, pad=10),
                        legend=dict(
                            orientation="h",
                            yanchor="bottom",
                            y=-0.2,
                            xanchor="center",
                            x=0.5
                        )
                    )
                else:
                    fig_aida = go.Figure()
                    fig_aida.update_layout(
                        title="<b>无AIDA数据</b>",
                        height=400,
                        plot_bgcolor=color_palette['panel'],
                        paper_bgcolor=color_palette['background'],
                        font_color=color_palette['text']
                    )
                
                # 显示图表
                display(fig_trend)
                with execution_output:
                    display(fig_execution)
                with aida_output:
                    display(fig_aida)

        # 绑定事件
        for dropdown in [project_dropdown, fv_dropdown, team_dropdown, 
                        tester_dropdown, aida_dropdown, week_dropdown, 
                        release_dropdown, status_dropdown]:
            dropdown.observe(update_dashboard, names='value')

        # 布局
        controls_row1 = HBox([project_dropdown, fv_dropdown, team_dropdown], 
                            layout=Layout(justify_content='space-between', width='100%', margin='10px 0'))
        controls_row2 = HBox([tester_dropdown, aida_dropdown, week_dropdown], 
                            layout=Layout(justify_content='space-between', width='100%', margin='10px 0'))
        controls_row3 = HBox([release_dropdown, status_dropdown], 
                            layout=Layout(justify_content='space-between', width='100%', margin='10px 0'))
        
        # 初始显示
        display(VBox([
            controls_row1, 
            controls_row2, 
            controls_row3, 
            trend_output, 
            execution_output, 
            aida_output
        ], layout=Layout(width='100%')))
        update_dashboard(None)

# 运行仪表板
create_dashboard()

In [ ]:
from ipywidgets import interact, Dropdown, HBox, VBox, Layout, Output, SelectMultiple, ToggleButtons, HTML
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display
import pandas as pd
import numpy as np

# Prepare filter options
projects = ['All'] + sorted(tdf['project'].dropna().unique().tolist())
fvs = ['All'] + sorted(tdf['fv'].dropna().unique().tolist())
teams = ['All'] + sorted(tdf['team'].dropna().unique().tolist())
testers = ['All'] + sorted(tdf['tester'].dropna().unique().tolist())
aidas = ['All'] + sorted(tdf['top_aida'].dropna().unique().tolist())
test_weeks = ['All'] + sorted(tdf['test_week'].dropna().unique().tolist())
releases = ['All'] + sorted(tdf['test_event'].dropna().unique().tolist())
run_statuses = ['All', 'Passed', 'Failed', 'Blocked', 'Planned']

# Color palette
color_palette = {
    'background': '#f5f5f5',
    'panel': '#ffffff',
    'text': '#333333',
    'primary': '#1f77b4',
    'secondary': '#ff7f0e',
    'success': '#2ca02c',
    'danger': '#d62728',
    'warning': '#ffbb78',
    'benchmark': '#ffcc00',
    'planned': '#7f7f7f'
}

# Create header
def create_header():
    display(HTML("""
    <div style="background-color:#2c3e50; color:white; padding:15px; margin-bottom:20px; border-radius:5px;">
        <h1 style="margin:0;">DTSV Test Quality Dashboard</h1>
        <p style="margin:5px 0 0; opacity:0.8;">Interactive visualization of test execution metrics</p>
    </div>
    """))

# Create sidebar
def create_sidebar():
    return HTML("""
    <div style="background-color:#ecf0f1; padding:15px; border-radius:5px; margin-right:20px; width:250px;">
        <h3 style="border-bottom:1px solid #bdc3c7; padding-bottom:10px;">Navigation</h3>
        <div style="margin-bottom:15px;">
            <h4 style="margin-bottom:5px;">Test Metrics</h4>
            <div style="padding-left:10px;">
                <p style="margin:3px 0;">• Pass Rate Trend</p>
                <p style="margin:3px 0;">• Failed Cases</p>
                <p style="margin:3px 0;">• Blocked Cases</p>
                <p style="margin:3px 0;">• Planned Cases</p>
            </div>
        </div>
        <div style="margin-bottom:15px;">
            <h4 style="margin-bottom:5px;">Filters Applied</h4>
            <div id="active-filters" style="padding-left:10px; font-size:0.9em;">
                <p style="margin:3px 0; color:#7f8c8d;">No filters selected</p>
            </div>
        </div>
    </div>
    """)

def create_status_chart(df, status, title, color):
    if df.empty:
        return go.Figure().update_layout(
            title=f"<b>{title}</b><br>No {status} test cases found",
            height=300,
            plot_bgcolor=color_palette['panel'],
            paper_bgcolor=color_palette['background']
        )
    
    # Group by function and tester
    grouped = df.groupby(['fv', 'tester']).size().reset_index(name='count')
    
    # Create sunburst chart for hierarchical visualization
    fig = px.sunburst(
        grouped,
        path=['fv', 'tester'],
        values='count',
        color_discrete_sequence=[color],
        title=f"<b>{title}</b>",
        height=400
    )
    
    fig.update_traces(
        textinfo='label+value',
        hovertemplate="<b>%{label}</b><br>Count: %{value}<extra></extra>",
        marker=dict(line=dict(width=0.5, color='white'))
    )
    
    fig.update_layout(
        margin=dict(t=60, l=0, r=0, b=0),
        plot_bgcolor=color_palette['panel'],
        paper_bgcolor=color_palette['background'],
        font_color=color_palette['text']
    )
    
    return fig

def create_dashboard():
    # Create header
    create_header()
    
    # Create sidebar and main content
    sidebar = create_sidebar()
    main_content = Output(layout=Layout(width='100%'))
    
    # Main layout
    main_layout = HBox([sidebar, main_content], layout=Layout(width='100%'))
    display(main_layout)
    
    with main_content:
        # Create control panel
        display(HTML("""
        <div style="background-color:#ffffff; padding:15px; border-radius:5px; margin-bottom:20px; box-shadow:0 2px 4px rgba(0,0,0,0.1);">
            <h3 style="margin-top:0; color:#2c3e50;">Filter Options</h3>
            <p style="color:#7f8c8d; margin-bottom:0;">Select filters to analyze test execution data</p>
        </div>
        """))
        
        # Create dropdowns
        project_dropdown = Dropdown(options=projects, value='All', description='Project:', 
                                  layout=Layout(width='300px'), style={'description_width': '80px'})
        fv_dropdown = Dropdown(options=fvs, value='All', description='Function:', 
                              layout=Layout(width='300px'), style={'description_width': '80px'})
        team_dropdown = Dropdown(options=teams, value='All', description='Team:', 
                                layout=Layout(width='300px'), style={'description_width': '80px'})
        tester_dropdown = Dropdown(options=testers, value='All', description='Tester:', 
                                  layout=Layout(width='300px'), style={'description_width': '80px'})
        week_dropdown = Dropdown(options=test_weeks, value='All', description='Test Week:', 
                                layout=Layout(width='300px'), style={'description_width': '80px'})
        release_dropdown = Dropdown(options=releases, value='All', description='Release:', 
                                   layout=Layout(width='300px'), style={'description_width': '80px'})

        # Output areas
        pass_rate_output = Output(layout=Layout(width='100%', margin='20px 0'))
        status_charts_output = Output(layout=Layout(width='100%', margin='20px 0'))
        details_output = Output(layout=Layout(width='100%', margin='20px 0'))

        # Update function
        def update_dashboard(change):
            with pass_rate_output:
                pass_rate_output.clear_output()
                with status_charts_output:
                    status_charts_output.clear_output()
                with details_output:
                    details_output.clear_output()
                
                # Get filter values
                project = project_dropdown.value
                fv = fv_dropdown.value
                team = team_dropdown.value
                tester = tester_dropdown.value
                test_week = week_dropdown.value
                release = release_dropdown.value
                
                # Filter data
                filtered = tdf.copy()
                if project != 'All':
                    filtered = filtered[filtered['project'] == project]
                if fv != 'All':
                    filtered = filtered[filtered['fv'] == fv]
                if team != 'All':
                    filtered = filtered[filtered['team'] == team]
                if tester != 'All':
                    filtered = filtered[filtered['tester'] == tester]
                if test_week != 'All':
                    filtered = filtered[filtered['test_week'] == test_week]
                if release != 'All':
                    filtered = filtered[filtered['test_event'] == release]
                
                # Update active filters display
                active_filters = []
                if project != 'All': active_filters.append(f"Project: {project}")
                if fv != 'All': active_filters.append(f"Function: {fv}")
                if team != 'All': active_filters.append(f"Team: {team}")
                if tester != 'All': active_filters.append(f"Tester: {tester}")
                if test_week != 'All': active_filters.append(f"Week: {test_week}")
                if release != 'All': active_filters.append(f"Release: {release}")
                
                if not active_filters:
                    filter_text = "No filters selected"
                else:
                    filter_text = "• " + "<br>• ".join(active_filters)
                
                display(HTML(f"""
                <script>
                    document.getElementById('active-filters').innerHTML = `{filter_text}`;
                </script>
                """))
                
                # 1. Pass Rate Trend (larger chart)
                if not filtered.empty:
                    # Weekly pass rate calculation
                    weekly_data = filtered.groupby('test_week').agg(
                        total_tests=('run_status', 'count'),
                        passed_tests=('run_status', lambda x: (x == 'Passed').sum()),
                        failed_tests=('run_status', lambda x: (x == 'Failed').sum()),
                        blocked_tests=('run_status', lambda x: (x == 'Blocked').sum()),
                        planned_tests=('run_status', lambda x: (x == 'Planned').sum()))
                    
                    weekly_data['pass_rate'] = (weekly_data['passed_tests'] / weekly_data['total_tests'] * 100).round(2)
                    weekly_data = weekly_data.reset_index()
                    
                    # Create larger trend chart
                    fig_trend = go.Figure()
                    
                    # Benchmark line
                    fig_trend.add_shape(
                        type='line',
                        x0=weekly_data['test_week'].min(),
                        x1=weekly_data['test_week'].max(),
                        y0=95,
                        y1=95,
                        line=dict(color=color_palette['benchmark'], width=3, dash='dot'),
                        name='Target (95%)'
                    )
                    
                    # Pass rate line
                    fig_trend.add_trace(go.Scatter(
                        x=weekly_data['test_week'],
                        y=weekly_data['pass_rate'],
                        mode='lines+markers+text',
                        name='Pass Rate',
                        line=dict(color=color_palette['primary'], width=4),
                        marker=dict(size=12, color=color_palette['primary']),
                        text=weekly_data['pass_rate'].apply(lambda x: f"{x}%"),
                        textposition="top center",
                        textfont=dict(size=12),
                        hoverinfo='text+x+y'
                    ))
                    
                    # Status breakdown area chart
                    fig_trend.add_trace(go.Scatter(
                        x=weekly_data['test_week'],
                        y=weekly_data['failed_tests'],
                        stackgroup='one',
                        name='Failed',
                        mode='lines',
                        line=dict(width=0.5, color=color_palette['danger']),
                        hoverinfo='y+name'
                    ))
                    
                    fig_trend.add_trace(go.Scatter(
                        x=weekly_data['test_week'],
                        y=weekly_data['blocked_tests'],
                        stackgroup='one',
                        name='Blocked',
                        mode='lines',
                        line=dict(width=0.5, color=color_palette['warning']),
                        hoverinfo='y+name'
                    ))
                    
                    fig_trend.add_trace(go.Scatter(
                        x=weekly_data['test_week'],
                        y=weekly_data['planned_tests'],
                        stackgroup='one',
                        name='Planned',
                        mode='lines',
                        line=dict(width=0.5, color=color_palette['planned']),
                        hoverinfo='y+name'
                    ))
                    
                    fig_trend.update_layout(
                        title="<b>Test Pass Rate Trend with Status Breakdown</b>",
                        title_font_size=20,
                        xaxis_title='Test Week',
                        yaxis_title='Pass Rate (%) / Test Count',
                        height=600,  # Larger chart size
                        hovermode="x unified",
                        plot_bgcolor=color_palette['panel'],
                        paper_bgcolor=color_palette['background'],
                        font=dict(size=14),
                        margin=dict(l=80, r=80, b=80, t=100, pad=10),
                        legend=dict(
                            orientation="h",
                            yanchor="bottom",
                            y=1.02,
                            xanchor="right",
                            x=1
                        )
                    )
                    
                    fig_trend.update_yaxes(range=[0, 105])
                    
                    display(fig_trend)
                    
                    # 2. Interactive status charts
                    with status_charts_output:
                        # Create a 1x3 grid for the status charts
                        display(HTML("<h3 style='margin-bottom:20px;'>Test Case Status Analysis</h3>"))
                        
                        # Get status-specific data
                        failed_cases = filtered[filtered['run_status'] == 'Failed']
                        blocked_cases = filtered[filtered['run_status'] == 'Blocked']
                        planned_cases = filtered[filtered['run_status'] == 'Planned']
                        
                        # Create charts
                        fig_failed = create_status_chart(
                            failed_cases, 
                            'Failed', 
                            'Failed Test Cases by Function & Tester', 
                            color_palette['danger']
                        )
                        
                        fig_blocked = create_status_chart(
                            blocked_cases, 
                            'Blocked', 
                            'Blocked Test Cases by Function & Tester', 
                            color_palette['warning']
                        )
                        
                        fig_planned = create_status_chart(
                            planned_cases, 
                            'Planned', 
                            'Planned Test Cases by Function & Tester', 
                            color_palette['planned']
                        )
                        
                        # Display in a grid
                        cols = HBox([
                            Output(layout=Layout(width='33%', margin='0 10px')),
                            Output(layout=Layout(width='33%', margin='0 10px')),
                            Output(layout=Layout(width='33%', margin='0 10px'))
                        ])
                        
                        display(cols)
                        
                        with cols.children[0]:
                            display(fig_failed)
                        with cols.children[1]:
                            display(fig_blocked)
                        with cols.children[2]:
                            display(fig_planned)
                    
                    # 3. Detailed tables (optional - can be shown in tooltips)
                else:
                    display(HTML("""
                    <div style="padding:20px; background-color:#f8d7da; color:#721c24; 
                                border-radius:5px; margin:20px 0;">
                        No data available for the selected filters.
                    </div>
                    """))

        # Bind events
        for dropdown in [project_dropdown, fv_dropdown, team_dropdown, 
                        tester_dropdown, week_dropdown, release_dropdown]:
            dropdown.observe(update_dashboard, names='value')

        # Layout controls
        controls_row1 = HBox([project_dropdown, fv_dropdown, team_dropdown], 
                           layout=Layout(justify_content='space-between', width='100%', margin='10px 0'))
        controls_row2 = HBox([tester_dropdown, week_dropdown, release_dropdown], 
                           layout=Layout(justify_content='space-between', width='100%', margin='10px 0'))
        
        # Initial display
        display(VBox([
            controls_row1, 
            controls_row2,
            pass_rate_output,
            status_charts_output,
            details_output
        ], layout=Layout(width='100%')))
        
        # Initial update
        update_dashboard(None)

# Run dashboard
create_dashboard()

In [ ]:
import plotly.graph_objects as go
import pandas as pd
import numpy as np
import re
from ipywidgets import widgets
from IPython.display import display, clear_output

def extract_english(text):
    if pd.isna(text) or text == '':
        return 'Unknown'
    english_words = re.findall(r'[a-zA-Z]+', str(text))
    return ' '.join(english_words) if english_words else 'Unknown' 


def preprocess_data(df):
    df_clean = df.copy()
    for col, val in [('test_week', '未知周'), ('project', '未知项目'), 
                    ('top_aida', '未知AIDA'), ('run_status', '未知状态')]:
        df_clean = df_clean[~df_clean[col].isin(['', val, None, np.nan])]
    df_clean['aida_english'] = df_clean['top_aida'].apply(extract_english)
    return df_clean

def create_aida_status_dashboard(df_clean, filters=None):
    filtered_data = df_clean.copy()
    
    if filters:
        project, tester, test_week, aida = filters
        filter_conditions = {
            'project': project, 
            'tester': tester, 
            'test_week': test_week, 
            'aida_english': aida
        }
        for col, val in filter_conditions.items():
            if val != '全部':
                filtered_data = filtered_data[filtered_data[col] == val]
    
    if filtered_data.empty:
        return None, "没有有效的项目数据可以显示。"
    
    fig = go.Figure()
    
    aida_status = filtered_data.groupby(['aida_english', 'run_status']).agg(
        测试数量=('id', 'count'),
        测试ID=('id', list),
        测试员=('tester', lambda x: list(set(x))),
        原始AIDA=('top_aida', lambda x: list(set(x))[0])
    ).reset_index()
    
    if aida_status.empty:
        return None, "没有有效的AIDA领域数据。"
    
    status_order = ['Passed', 'Failed', 'Requires Attention', 'In Progress', 'Planned']
    status_colors = {
        'Passed': '#4CAF50', 'Failed': '#F44336', 'Requires Attention': '#FF9800',
        'In Progress': '#2196F3', 'Planned': '#9C27B0'
    }
    
    # Calculate total counts per AIDA for percentage calculation
    aida_totals = aida_status.groupby('aida_english')['测试数量'].sum().to_dict()
    
    for status in status_order:
        status_data = aida_status[aida_status['run_status'] == status]
        if not status_data.empty:
            hover_texts = []
            for _, row in status_data.iterrows():
                ids = row['测试ID'][:5]
                testers = row['测试员'][:3]
                total = aida_totals[row['aida_english']]
                percentage = row['测试数量'] / total * 100
                
                hover_text = f"<b>{row['原始AIDA']} - {row['run_status']}</b><br>"
                hover_text += f"测试数量: {row['测试数量']} ({percentage:.1f}%)<br>"
                hover_text += f"测试ID: {', '.join(str(id) for id in ids)}"
                if len(row['测试ID']) > 5:
                    hover_text += f" 等{len(row['测试ID'])}个"
                hover_text += f"<br>测试员: {', '.join(str(t) for t in testers)}"
                if len(row['测试员']) > 3:
                    hover_text += f" 等{len(row['测试员'])}人"
                
                hover_texts.append(hover_text)
            
            fig.add_trace(go.Bar(
                x=status_data['aida_english'],
                y=status_data['测试数量'],
                name=status,
                marker_color=status_colors.get(status, '#9E9E9E'),
                hovertemplate='%{hovertext}<extra></extra>',
                hovertext=hover_texts,
                width=0.8  # 只保留宽度设置，移除所有文字显示相关参数
            ))
    
    title_text = "所有项目 - AIDA领域测试状态" if filters[0] == '全部' else f"项目: {filters[0]} - AIDA领域测试状态"
    
    fig.update_layout(
        height=800,
        width=1200,
        barmode='stack',
        title=dict(
            text=f"{title_text}<br><sup>项目: {filters[0]} | 测试员: {filters[1]} | 测试周: {filters[2]} | AIDA: {filters[3]}</sup>",
            x=0.5,
            y=0.95,
            font=dict(size=16)
        ),
        template='plotly_white',
        margin=dict(t=120)
    )
    fig.update_xaxes(title_text="AIDA领域", tickangle=-45)
    fig.update_yaxes(
        title_text="测试数量", 
        showticklabels=True,
        showgrid=True
    )
    
    return fig, None

def create_test_dashboard(df):
    df_clean = preprocess_data(df)
    
    required_columns = ['id', 'test_week', 'project', 'top_aida', 'run_status', 'tester']
    if not all(col in df_clean.columns for col in required_columns):
        print(f"错误：数据中缺少必要的列")
        return
    
    # 获取唯一值用于下拉菜单
    dropdowns = {}
    for col, desc in [('project', '项目'), ('tester', '测试员'), 
                     ('test_week', '测试周'), ('aida_english', 'AIDA')]:
        values = sorted([p for p in df_clean[col].unique() if pd.notna(p)])
        dropdowns[col] = widgets.Dropdown(
            options=['全部'] + values, value='全部', description=f'{desc}:',
            layout={'width': '180px'}
        )
    
    output = widgets.Output()
    
    def update_charts(change):
        filters = (
            dropdowns['project'].value,
            dropdowns['tester'].value,
            dropdowns['test_week'].value,
            dropdowns['aida_english'].value
        )
        
        with output:
            clear_output(wait=True)  # 清除旧输出
            fig, error_msg = create_aida_status_dashboard(df_clean, filters)
            if fig:
                display(fig)
            else:
                print(error_msg)
    
    # 为所有下拉菜单添加观察者
    for dropdown in dropdowns.values():
        dropdown.observe(update_charts, names='value')
    
    # 创建标题
    title = widgets.HTML("<h2 style='text-align:center; margin-bottom:5px'>测试用例执行状态看板</h2>")
    
    # 将筛选条件放在一行中
    filter_container = widgets.HBox(list(dropdowns.values()), 
                                   layout=widgets.Layout(justify_content='center', 
                                                        margin='0 0 10px 0'))
    
    # 使用VBox将所有元素垂直排列
    dashboard = widgets.VBox([
        title,
        filter_container,
        output
    ], layout=widgets.Layout(align_items='center'))
    
    display(dashboard)
    update_charts(None)  # 初始显示

# 使用你的数据框调用函数
create_test_dashboard(tdf)

In [ ]:
import plotly.graph_objects as go
import pandas as pd
import numpy as np
import re
from ipywidgets import widgets, Layout
from IPython.display import display, clear_output

def extract_english(text):
    if pd.isna(text) or text == '':
        return 'Unknown'
    english_words = re.findall(r'[a-zA-Z]+', str(text))
    return ' '.join(english_words) if english_words else 'Unknown'

def preprocess_data(df):
    df_clean = df.copy()
    for col, val in [('test_week', '未知周'), ('project', '未知项目'), 
                    ('top_aida', '未知AIDA'), ('run_status', '未知状态')]:
        df_clean = df_clean[~df_clean[col].isin(['', val, None, np.nan])]
    df_clean['aida_english'] = df_clean['top_aida'].apply(extract_english)
    return df_clean

def create_aida_status_dashboard(df_clean, filters=None):
    filtered_data = df_clean.copy()
    
    if filters:
        project, tester, test_week, aida = filters
        filter_conditions = {
            'project': project, 
            'tester': tester, 
            'test_week': test_week, 
            'aida_english': aida
        }
        for col, val in filter_conditions.items():
            if val != '全部':
                filtered_data = filtered_data[filtered_data[col] == val]
    
    if filtered_data.empty:
        return None, "没有有效的项目数据可以显示。"
    
    # 创建主图和右侧信息面板的容器
    main_fig = go.FigureWidget()
    info_panel = widgets.Output()
    
    aida_status = filtered_data.groupby(['aida_english', 'run_status']).agg(
        测试数量=('id', 'count'),
        测试ID=('id', list),
        测试员=('tester', lambda x: list(set(x))),
        原始AIDA=('top_aida', lambda x: list(set(x))[0])
    ).reset_index()
    
    if aida_status.empty:
        return None, "没有有效的AIDA领域数据。"
    
    status_order = ['Passed', 'Failed', 'Requires Attention', 'In Progress', 'Planned']
    status_colors = {
        'Passed': '#4CAF50', 'Failed': '#F44336', 'Requires Attention': '#FF9800',
        'In Progress': '#2196F3', 'Planned': '#9C27B0'
    }
    
    # Calculate total counts per AIDA for percentage calculation
    aida_totals = aida_status.groupby('aida_english')['测试数量'].sum().to_dict()
    
    for status in status_order:
        status_data = aida_status[aida_status['run_status'] == status]
        if not status_data.empty:
            hover_texts = []
            for _, row in status_data.iterrows():
                ids = row['测试ID'][:5]
                testers = row['测试员'][:3]
                total = aida_totals[row['aida_english']]
                percentage = row['测试数量'] / total * 100
                
                hover_text = f"<b>{row['原始AIDA']} - {row['run_status']}</b><br>"
                hover_text += f"测试数量: {row['测试数量']} ({percentage:.1f}%)<br>"
                hover_text += f"测试ID: {', '.join(str(id) for id in ids)}"
                if len(row['测试ID']) > 5:
                    hover_text += f" 等{len(row['测试ID'])}个"
                hover_text += f"<br>测试员: {', '.join(str(t) for t in testers)}"
                if len(row['测试员']) > 3:
                    hover_text += f" 等{len(row['测试员'])}人"
                
                hover_texts.append(hover_text)
            
            main_fig.add_trace(go.Bar(
                x=status_data['aida_english'],
                y=status_data['测试数量'],
                name=status,
                marker_color=status_colors.get(status, '#9E9E9E'),
                hovertemplate='%{hovertext}<extra></extra>',
                hovertext=hover_texts,
                customdata=status_data[['原始AIDA', 'run_status', '测试数量', '测试ID', '测试员']].values,
                width=0.8
            ))
    
    title_text = "所有项目 - AIDA领域测试状态" if filters[0] == '全部' else f"项目: {filters[0]} - AIDA领域测试状态"
    
    main_fig.update_layout(
        height=800,
        width=1150,  # 减小宽度为右侧面板留空间
        barmode='stack',
        title=dict(
            text=f"{title_text}<br><sup>项目: {filters[0]} | 测试员: {filters[1]} | 测试周: {filters[2]} | AIDA: {filters[3]}</sup>",
            x=0.5,
            y=0.95,
            font=dict(size=16)
        ),
        template='plotly_white',
        margin=dict(t=120, r=300)  # 右侧留出空间
    )
    main_fig.update_xaxes(title_text="AIDA领域", tickangle=-45)
    main_fig.update_yaxes(
        title_text="测试数量", 
        showticklabels=True,
        showgrid=True
    )
    
    # 点击事件处理函数
    def on_click(trace, points, selector):
        with info_panel:
            clear_output(wait=True)
            if points.point_inds:
                point_index = points.point_inds[0]
                data = trace.customdata[point_index]
                aida, status, count, ids, testers = data
                
                # 计算百分比
                total = aida_totals.get(trace.x[point_index], 1)
                percentage = (count / total) * 100
                
                # 创建HTML显示内容
                html = widgets.HTML(
                    value=f"""
                    <div style="padding:10px; background:#f9f9f9; border-radius:5px; border-left:5px solid {status_colors.get(status, '#9E9E9E')};">
                        <h3 style="margin-top:0; color:#333;">{aida} - {status}</h3>
                        <p><b>测试数量:</b> {count} ({percentage:.1f}%)</p>
                        <p><b>测试ID:</b> {', '.join(map(str, ids[:5]))}{' 等' if len(ids)>5 else ''}</p>
                        <p><b>测试员:</b> {', '.join(testers[:3])}{' 等' if len(testers)>3 else ''}</p>
                    </div>
                    """,
                    layout=Layout(width='280px', margin='10px 0')
                )
                display(html)
    
    # 为每个柱状图添加点击事件
    for trace in main_fig.data:
        trace.on_click(on_click)
    
    # 创建右侧信息面板标题
    info_title = widgets.HTML(
        value="<h3 style='margin-bottom:5px;'>详细信息</h3><p style='color:#666; font-size:0.9em; margin-top:0;'>点击柱状图查看详情</p>",
        layout=Layout(width='280px')
    )
    
    # 将图表和信息面板组合在一起
    dashboard = widgets.HBox([
        main_fig,
        widgets.VBox([info_title, info_panel], 
                     layout=Layout(width='300px', margin='120px 0 0 0'))
    ], layout=Layout(justify_content='center'))
    
    return dashboard, None

def create_test_dashboard(df):
    df_clean = preprocess_data(df)
    
    required_columns = ['id', 'test_week', 'project', 'top_aida', 'run_status', 'tester']
    if not all(col in df_clean.columns for col in required_columns):
        print(f"错误：数据中缺少必要的列")
        return
    
    # 获取唯一值用于下拉菜单
    dropdowns = {}
    for col, desc in [('project', '项目'), ('tester', '测试员'), 
                     ('test_week', '测试周'), ('aida_english', 'AIDA')]:
        values = sorted([p for p in df_clean[col].unique() if pd.notna(p)])
        dropdowns[col] = widgets.Dropdown(
            options=['全部'] + values, value='全部', description=f'{desc}:',
            layout={'width': '180px'}
        )
    
    output = widgets.Output()
    
    def update_charts(change):
        filters = (
            dropdowns['project'].value,
            dropdowns['tester'].value,
            dropdowns['test_week'].value,
            dropdowns['aida_english'].value
        )
        
        with output:
            clear_output(wait=True)  # 清除旧输出
            fig, error_msg = create_aida_status_dashboard(df_clean, filters)
            if fig:
                display(fig)
            else:
                print(error_msg)
    
    # 为所有下拉菜单添加观察者
    for dropdown in dropdowns.values():
        dropdown.observe(update_charts, names='value')
    
    # 创建标题
    title = widgets.HTML("<h2 style='text-align:center; margin-bottom:5px'>测试用例执行状态看板</h2>")
    
    # 将筛选条件放在一行中
    filter_container = widgets.HBox(list(dropdowns.values()), 
                                   layout=widgets.Layout(justify_content='center', 
                                                        margin='0 0 10px 0'))
    
    # 使用VBox将所有元素垂直排列
    dashboard = widgets.VBox([
        title,
        filter_container,
        output
    ], layout=widgets.Layout(align_items='center'))
    
    display(dashboard)
    update_charts(None)  # 初始显示

# 使用你的数据框调用函数
create_test_dashboard(tdf)

In [ ]:
import plotly.express as px
# 将非标准空格替换为标准空格
px.defaults.template = "simple_white"

# 使用 tdf 数据框
# 同时也将非标准空格替换为标准空格
fig = px.scatter(tdf.groupby(["test_week", "fv", "run_status"]).count().reset_index().sort_values("test_week"), x="test_week", y="fv", color="run_status", size="test_id")
fig.update_layout(scattermode="group", scattergap=1)
fig.show()

In [ ]:
# 对数据进行分组，并计算每个分组的测试数量
grouped_data = tdf.groupby(["test_week", "fvp", "fv", "run_status"]).size().reset_index(name='test_count')

extract_pattern = r'(\d{2})-(\d{2})'
extracted_sort_keys = grouped_data['test_week'].str.extract(extract_pattern)

# 如果提取成功，创建排序用的数字列
if not extracted_sort_keys.isnull().all().all(): # 检查是否至少提取到了一些内容
    grouped_data['sort_year'] = pd.to_numeric(extracted_sort_keys[0], errors='coerce') # 提取的年份转为数字
    grouped_data['sort_week'] = pd.to_numeric(extracted_sort_keys[1], errors='coerce') # 提取的周数转为数字
else:
    # 如果提取失败（格式不匹配），可以设置默认值或引发错误
    # 这里我们先假设总能提取到，如果报错再处理
    print("警告：未能从 test_week 提取排序键，排序可能不正确。")
    # 可以添加备用排序逻辑，例如按原始 test_week 排序
    grouped_data['sort_year'] = 0 # 示例默认值
    grouped_data['sort_week'] = 0 # 示例默认值

grouped_data = grouped_data.sort_values(["sort_year", "sort_week", "fvp", "fv"])

fig = px.scatter(
    grouped_data,
    x="test_week",
    y="fv",
    color="run_status",
    size="test_count",
    hover_data=["fvp", "fv", "test_week", "run_status", "test_count"],
    title="按周、FVP 和功能分类的测试状态"
)

# 更新布局
fig.update_layout(
    scattermode="group",
    scattergap=0.7,
    xaxis_title="测试周",
    yaxis_title="功能 (按 FVP 排序分组)",
    # 强制 X 轴按排序后的 test_week 顺序显示
    xaxis={'categoryorder':'array', 'categoryarray': grouped_data['test_week'].unique()},
    # 强制 Y 轴按排序后的 fv 顺序显示
    yaxis={'categoryorder':'array', 'categoryarray': grouped_data['fv'].unique()},
    width=1000,
    height=800
)

# 显示图表
fig.show()

# 可选：删除临时的排序辅助列
# grouped_data = grouped_data.drop(columns=['sort_year', 'sort_week'])

In [ ]:
import plotly.express as px
px.defaults.template = "simple_white"
 
# 创建分组数据框
grouped_data = tdf.groupby(["test_week", "fvp", "fv", "run_status"]).count().reset_index()
 
# 使用提取模式
extract_pattern = r'(\d{2})-(\d{2})'
extracted_sort_keys = grouped_data['test_week'].str.extract(extract_pattern)
 
# 如果提取成功，创建排序用的数字列
if not extracted_sort_keys.isnull().all().all():  # 检查是否至少提取到了一些内容
    grouped_data['sort_year'] = pd.to_numeric(extracted_sort_keys[0], errors='coerce')  # 提取的年份转为数字
    grouped_data['sort_week'] = pd.to_numeric(extracted_sort_keys[1], errors='coerce')  # 提取的周数转为数字
else:
    print("警告：未能从 test_week 提取排序键，排序可能不正确。")
    grouped_data['sort_year'] = 0
    grouped_data['sort_week'] = 0
 
# 按年和周排序数据
grouped_data = grouped_data.sort_values(["sort_year", "sort_week", "fvp", "fv"])
 
# 保存排序后的唯一值列表用于坐标轴排序
sorted_weeks = grouped_data['test_week'].unique()
sorted_fvs = grouped_data.sort_values(['fvp', 'fv'])['fv'].unique()
 
# 创建颜色映射字典
color_map = {
    "Passed": "green",
    "Failed": "red",
    "Requires Attention": "yellow"  # 如果实际值是"required"或其他，请相应调整
}
 
# 创建气泡图 (使用px.scatter但优化气泡效果)
fig = px.scatter(
    grouped_data,
    x="test_week",
    y="fv",
    color="run_status",
    size="id",
    size_max=40,  # 增大气泡的最大尺寸
    opacity=0.85,  # 调整透明度使重叠部分可见
    hover_data=["fvp", "test_week", "run_status", "id"],
    labels={"test_week": "测试周", "fv": "功能", "run_status": "运行状态", "id": "测试数量"},
    color_discrete_map=color_map  # 应用自定义颜色映射
)
 
# 更新布局
fig.update_layout(
    width=1300,  # 宽度
    height=900,  # 高度
    xaxis={
        'categoryorder':'array',
        'categoryarray': sorted_weeks,
        'tickangle': -45  # 倾斜x轴标签以便更好地显示
    },
    yaxis={
        'categoryorder':'array',
        'categoryarray': sorted_fvs
    },
    xaxis_title="测试周",
    yaxis_title="功能 (按 FVP 分组)",
    title="按周和功能分类的测试状态",
    showlegend=True,
    legend_title_text="运行状态",
    # 添加网格线以提高可读性
    xaxis_showgrid=True,
    yaxis_showgrid=True,
    plot_bgcolor='rgba(248, 248, 250, 0.5)'  # 浅色背景
)
 
# 增强气泡效果
fig.update_traces(
    marker=dict(
        sizemode='area'  # 确保气泡大小按面积比例
    )
)
 
fig.show()